In [ ]:
from langgraph.graph import StateGraph, START,END
import os
from typing import TypedDict, Annotated  
from pydantic import BaseModel, Field


from langchain_openai import ChatOpenAI
import operator


from dotenv import load_dotenv
load_dotenv()


In [ ]:


model = ChatOpenAI(
    model="google/gemini-2.5-flash-lite",
    max_tokens=1000,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


In [ ]:
class EvaluationSchema(BaseModel):
    feedback: str= Field(description = 'Detailed Feedback for the essay')
    score :int = Field(description='score out of 10', ge=0 , le=10)

In [ ]:
structured_model= model.with_structured_output(EvaluationSchema)

In [ ]:
essay= """
# Role of AI in Pakistan

Artificial Intelligence (AI) is one of the most important technologies of the modern world. It enables computers and machines to perform tasks that normally require human intelligence, such as learning, problem-solving, and decision-making. For Pakistan, AI offers significant opportunities for economic growth, better public services, and technological development.

One major area where AI can benefit Pakistan is **education**. AI-powered tools can provide students with personalized learning, explain difficult concepts, and help teachers prepare educational material. This can be especially useful for students who have limited access to quality educational resources.

AI can also improve **healthcare** in Pakistan. It can assist doctors in analyzing medical images, managing patient records, and identifying patterns that may help with early diagnosis. AI-based systems can also make hospital management more efficient.

Another important field is **agriculture**. Pakistan depends heavily on agriculture, but farmers face problems such as water shortages, crop diseases, and unpredictable weather. AI can analyze weather and soil data, detect crop diseases, and help farmers use water and fertilizers more efficiently.

The **IT industry** is another major opportunity. Pakistan has a large number of young people interested in technology and freelancing. By developing skills in AI, machine learning, and data science, Pakistani professionals can provide services to international companies and contribute to the country's technology exports.

AI can also improve **banking and government services**. Banks can use AI to detect fraud and provide faster customer support, while government institutions can use it to analyze data and improve public services.

However, Pakistan also faces challenges in adopting AI. These include a lack of advanced technical skills, limited research funding, insufficient digital infrastructure, and concerns about data privacy and cybersecurity. There is also a risk that automation may replace some repetitive jobs.

Therefore, Pakistan needs to invest in **AI education, research, digital infrastructure, and responsible technology policies**. Universities, businesses, and the government should work together to develop skilled professionals and encourage innovation.

In conclusion, AI has great potential to transform Pakistan's economy and society. If Pakistan invests in the right skills, infrastructure, and research, AI can create new opportunities, improve important services, and help the country compete in the global digital economy.

"""

In [ ]:
class essayState (TypedDict):
    essay: str 
    language_feedback :str
    analysis_feedback : str
    clarity_feedback: str
    overall_feedback : str
    individual_scores: Annotated[list[int], operator.add]
    avg_score : float

In [ ]:
def evaluate_language(state: essayState):
    prompt= f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output =structured_model.invoke(prompt)
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def evaluate_analysis(state: essayState):
    prompt= f'Evaluate the debth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output =structured_model.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def evaluate_thought(state: essayState):
    prompt= f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output =structured_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}


In [ ]:
# def final_evaluation(state: essayState):
#     prompt= f'Based on the folllowing feedbacks create a summerized feedback \n language feedback'
#     output =structured_model.invoke(prompt)
#     return {'clarity_feedback': output['feedback'], 'individual_scores': output['score']}

In [ ]:
def final_evaluation(state: essayState):
    language_fb = state.get('language_feedback')
    analysis_fb = state.get("analysis_feedback")
    clarity_fb = state.get("clarity_feedback")

    prompt = f"""You are the final evaluator. Based on the following evaluations of the essay, 
    provide a synthesized overall feedback and a final consolidated score (1-100).

    --- Language Feedback ---
    {language_fb}

    --- analysis Feedback ---
    {analysis_fb}

    --- clarity Feedback ---
    {clarity_fb}
    """
    overall_feedback = model.invoke(prompt).content

    avg_score=sum(state['individual_scores'])/len(state['individual_scores'])
    return {'overall_feedback':overall_feedback, 'avg_score' : avg_score}


In [ ]:
graph = StateGraph(essayState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)

graph.add_node('final_evaluation',final_evaluation)

graph.add_edge(START,'evaluate_language' )
graph.add_edge(START,'evaluate_analysis' )
graph.add_edge(START,'evaluate_thought' )
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [ ]:
initial_state={
    'essay': essay
}

workflow.invoke(initial_state)